# Lab08 — من التسلسل إلى القرابة: مختبر Boltz-2 (P14)

**خطة التطوير §الربع الثاني، P14** — المقارنة ثلاثية الطبقات لتسعير القرابة.

## السؤال المحوري

كم يكلّف ربط الجزيء بالمستقبل من منظور **ثلاث طبقات تسعير** مختلفة الجهد والثقة — ومتى نثق بأي طبقة ولماذا؟

| الطبقة | المحرك | الجهد | المخرج | الثقة الموثقة |
|---|---|---|---|---|
| 1 سريعة | نموذجنا ML v0.12.0 (Morgan تكرارات + RF) | أجزاء ثانية، CPU | pIC50 ± CI95 | ترتيب داخل السلالة؛ قياس P12 الخارجي وثّق سقفها |
| 2 فيزيائية | AutoDock Vina على 7VH8 | دقائق، CPU | ΔG kcal/mol → pIC50 | درجات تعليمية؛ مرجع إعادة الربط −8.28 |
| **3 مرجعية** | **Boltz-2 co-folding** (هذا الدفتر) | **~20 ثانية، GPU** | **log10(IC50[μM]) + بنية** | رأس تقارب مدرَّب على FEP+/CASP16 — رخصة MIT |

## حقائق Boltz-2 (من المستودع الرسمي jwohlwend/boltz)

- الكود والنماذج **رخصة MIT** كاملة، والاستشهاد: Passaro et al., bioRxiv 2025.06.14.659707.
- رأس التقارب يُخرج حقلين:
  - `affinity_pred_value` = **log10(IC50)** مع IC50 بوحدة **ميكرومولار (μM)**
    → التحويل: IC50[μM] = 10^(value) و pIC50 = **6 − value**.
  - `affinity_probability_binary` ∈ [0..1] — احتمال أن الجزيء مرتبط أصلاً (فرز اكتشاف).
- يشغل GPU بذاكرة ≥ 16GB — لهذا يعمل هذا الدفتر على **Colab** (T4/L4/A100)،
  ومخرجُه بيانٌ مجمّد `lab08-manifest.json` يستهلكه محلياً
  `core/clients/boltz2_client.py` ليعاد بناء الجدول الثلاثي بالكامل.

> **المبدأ الحاكم للمشروع:** لا تُختلق قيمة واحدة. الطبقة الثالثة بلا تشغيل حقيقي
> تبقى `unavailable_offline` صادقة في المجمّد المحلي — لا أرقام منسوخة من أوراق.


## 0) التشغيل

1. افتح الدفتر على Google Colab: **Runtime → Change runtime type → T4 GPU**.
2. شغّل الخلايا بالترتيب (الأولى تثبت `boltz`).
3. عند الانتهاء نزّل `lab08-manifest.json` من متصفح الملفات، وضعه في المستودع
   على `data/boltz2/lab08-manifest.json`، ثم أعد محلياً:
   `python3 scripts/evaluate_three_tier_p14.py` — فيُعاد بناء الجدول الثلاثي بالكامل.


In [ ]:
!pip install -q boltz
import boltz  # تحقق التثبيت
print("boltz version:", boltz.__version__ if hasattr(boltz, "__version__") else "installed")
import torch
assert torch.cuda.is_available(), "فعّل GPU: Runtime → Change runtime type"
print("GPU:", torch.cuda.get_device_name(0))

## 1) المدخلات — المستقبل والليغاندان المرجعيان

- **المستقبل**: سلسلة A من بنية 7VH8 (Mpro — نفس هدف بيانات التدريب ومستقبل
  الطبقة الثانية) — التسلسل الحقيقي 306 أحماض من RCSB FASTA.
- **الليغاندان** (نفس لوحة التجميد المحلي):
  1. **نيرماتريلڤير** (PF-07321332) — المرجع التجريبي ChEMBL: وسيط IC50 إنزيمية 31 nM (pIC50 ≈ 7.51)، انتشار السجلات 0.79–75 nM.
  2. **PF-00835231** — الليغاند المشتبك في 7VH8 نفسه: IC50 = 8.0 nM (pIC50 = 8.10، اختبار CHEMBL4810314).

البصمة sha256 للتسلسل تُحسب أدناه وتُجمَّد في البيان — أي تعديل للتسلسل يُكتشف تلقائياً.

In [ ]:
import hashlib

MPRO_SEQ = "SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQ"
LIGANDS = {
    "nirmatrelvir": "CC(C)(C)[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC1=CN=CS1)NC(=O)C(F)(F)F)C(=O)N1CC[C@@H](C1)C#N",
    "pf00835231": "NC(=N)N[C@@H]1[C@H]2CC[C@@H](CN2C1=O)NC(=O)[C@@H](NC(=O)[C@@H](NC(=O)C(F)(F)F)CC2CCCCC2)C(C)C",
}
assert len(MPRO_SEQ) == 306, f"طول غير متوقع: {len(MPRO_SEQ)}"
SEQ_SHA = hashlib.sha256(MPRO_SEQ.encode()).hexdigest()
print("Mpro 7VH8 chain A — 306 aa ✓ | sha256:", SEQ_SHA[:16], "…")

## 2) كتابة ملفات إدخال Boltz (YAML)

صيغة التنبؤ بالتقارب الوثيقة رسمياً: تسلسل البروتين + ليغاند SMILES + خاصية
`affinity` مع `binder` يشير إلى معرّف الليغاند.

In [ ]:
from pathlib import Path

WORK = Path("lab08_work"); WORK.mkdir(exist_ok=True)
for name, smi in LIGANDS.items():
    yaml_text = (
        "sequences:\n"
        "  - protein:\n"
        "      id: A\n"
        f"      sequence: {MPRO_SEQ}\n"
        "  - ligand:\n"
        "      id: L\n"
        f"      smiles: {smi}\n"
        "properties:\n"
        "  - affinity:\n"
        "      binder: L\n"
    )
    (WORK / f"{name}.yaml").write_text(yaml_text)
    print("كُتب:", WORK / f"{name}.yaml")
print(WORK / "nirmatrelvir.yaml" .read_text())

## 3) التشغيل الفعلي — co-folding + رأس التقارب

`--use_msa_server` يطلب MSA من خادم ColabFold العام. لو تعطّل الخادم أعد
الخلية — التشغيل ~دقائق قليلة لكل معقد على T4 (المعقد 306 + ليغاند صغير).

In [ ]:
import subprocess, time

for name in LIGANDS:
    t0 = time.time()
    cmd = [
        "boltz", "predict", str(WORK / f"{name}.yaml"),
        "--out_dir", str(WORK),
        "--use_msa_server",
        "--seed", "0",
    ]
    print("أمر التشغيل:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"اكتمل {name} في {time.time()-t0:.0f}s — رمز الخروج {result.returncode}")
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"فشل تشغيل {name} — راجع الرسالة أعلاه")

## 4) قراءة المخرجات وكتابة البيان المجمّد

مخرجات `boltz` تتغير أسماء ملفاتها بين الإصدارات — المُحلِّل أدناه دفاعي:
يتفحص كل JSON في مجلد التنبؤات ويسحب حقول التقارب والثقة الموجودة فعلاً،
ولا يخمّن شيئاً. الميتاداتا (الإصدار/العتاد/البذرة/أمر التشغيل) تُجمَّد معها.

In [ ]:
import json, glob, platform, datetime, torch

def find_affinity_files(out_root: Path):
    files = sorted(out_root.rglob("*affinity*.json")) + sorted(out_root.rglob("*affinity*.cif"))
    return [f for f in files if f.suffix == ".json"]

def parse_prediction(out_root: Path, name: str):
    """سحب دفاعي: كل JSON يحتوي حقول affinity يحتسب مرشحاً."""
    preds = []
    for f in find_affinity_files(out_root):
        data = json.loads(f.read_text())
        # قد يكون الحقل أعلى المستوى أو تحت مفتاح-قاموس
        candidates = [data] + [v for v in data.values() if isinstance(v, dict)]
        for cand in candidates:
            if "affinity_pred_value" in cand:
                pred = {
                    "affinity_pred_value": float(cand["affinity_pred_value"]),
                    "affinity_probability_binary": cand.get("affinity_probability_binary"),
                }
                for k in ("iptm", "ptm", "complex_plddt", "ligand_iptm", "ligand_plddt"):
                    if k in cand:
                        pred[k] = cand[k]
                pred["_file"] = str(f.relative_to(out_root))
                preds.append(pred)
                break
    assert preds, f"لا مخرجات تقارب لـ{name} — افحص مجلد {out_root}"
    return preds[0]  # البذرة 0 ببذرة واحدة

predictions, manifest_predictions = [], []
for name, smi in LIGANDS.items():
    pred = parse_prediction(WORK, name)
    value = pred["affinity_pred_value"]
    ic50_um = 10 ** value
    pic50 = 6.0 - value
    print(f"{name}: affinity_pred_value={value:.3f} → IC50={ic50_um:.3f} μM → pIC50={pic50:.2f}")
    entry = {
        "smiles": smi,
        "affinity_pred_value": value,
        "affinity_probability_binary": pred.get("affinity_probability_binary"),
        "seed": 0,
        "source": f"boltz predict {name}.yaml --use_msa_server --seed 0",
    }
    for k in ("iptm", "ptm", "complex_plddt", "ligand_iptm", "ligand_plddt"):
        if k in pred:
            entry[k] = pred[k]
    manifest_predictions.append(entry)

manifest = {
    "schema": "pharma-sdl.boltz2-affinity.v1",
    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "model": {
        "name": "boltz-2",
        "version": getattr(boltz, "__version__", "unknown"),
        "license": "MIT",
        "source_url": "https://github.com/jwohlwend/boltz",
        "citation": "Passaro et al., bioRxiv 2025.06.14.659707",
    },
    "hardware": {"device": torch.cuda.get_device_name(0), "colab": True},
    "receptor": {
        "name": "SARS-CoV-2 Mpro (3C-like proteinase)",
        "pdb_reference": "7VH8",
        "sequence_sha256": SEQ_SHA,
    },
    "predictions": manifest_predictions,
}
out_manifest = Path("lab08-manifest.json")
out_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print("→ البيان المجمّد:", out_manifest.resolve())
print("نزّله وضعه في المستودع: data/boltz2/lab08-manifest.json")

## 5) الدرس العربي — متى نثق بأي طبقة ولماذا

### الجدول الثلاثي (قارنه بالمجمّد المحلي)

افتح `docs/audit/model-three-tier-p14-results.json` بعد إعادة سكربت التجميد
وسترى الصفوف الثلاثة لكل ليغاند. القواعد الحاكمة للمشروع:

1. **قاعدة القمع**: الطبقة 1 تفرز **الآلاف** (أجزاء ثانية) ← الطبقة 2 تصفي
   **المئات** (دقائق) ← الطبقة 3 تُسعّر **العشرات** (~20 ثانية GPU لكل معقد) ←
   التجربة الحقيقية تحكم. كل طبقة رخيصة تسبق أغلى منها، ولا طبقة تتجاوز حكم
   طبقة أعلى منها.
2. **الثقة الموثقة لكل طبقة**:
   - ML (طبقة 1): ممتاز لـ**ترتيب** مرشحين داخل سلالته الكيميائية (Mpro) —
     لكن قياس P12 الخارجي (TDC) وثّق أن قيمه المطلقة خارج سلالته تتقلص نحو
     وسط التدريب (انضغاط انحداري) — لا تستخدمها للحكم النهائي.
   - Vina (طبقة 2): تعطي **موضع الربط** وطاقة خشنة مفيدة، لكنها تعليمية
     (انحياز معلوم)، وتعجز مبدئياً عن الليغاندات **التساهمية** مثل
     نيرماتريلڤير (رأس نيتريل) لأن دالة التسجيل غير تساهمية — مرجع إعادة
     الربط الذاتي (−8.28 kcal/mol لـ7VH8) هو شهادة صحة الصندوق وحده.
   - Boltz-2 (طبقة 3): رأس تقارب مدرَّب على مخرجات فيزيائية (FEP+ Pearson
     R=0.66، المرتبة 1 في تحدي CASP16 للقرابة) ويعطي **بنية + قرابة معاً** —
     هو الحكم المرجعي على الناجين، لكنه لا يعمل بلا GPU ولا بلا شبكة،
     وتكلفته أعلى بأوامر من الطبقتين الأوليين.
3. **قاعدة التعارض**: عند |ΔpIC50| > 2 بين طبقتين لا «تصويت طبقات» — الأقرب
   للمرجع التجريبي يكسب في هذه اللوحة، والتعارض نفسه نتيجة تُنشر وتُحلّل
   (السبب المتكرر: مرونة المستقبل مفقودة في الطبقتين 1 و2، أو ليغاند تساهمي
   أمام دالة تسجيل غير تساهمية).
4. **حدود Boltz-2 المعترف بها** (من ورقته): دقته التكرارية بين البذور
   محدودة، ورأس التقارب حساس لجودة MSA، وتدريبه على IC50/Ki/Kd مختلطة —
   لذلك نجمّد البذرة (0) والنسخة في البيان، وننشر كل شيء كما هو.

### ماذا تعني القيم التي حصلت عليها؟

- `affinity_pred_value` سالب أكثر = ربط أقوى (IC50 أصغر).
- مثال: القيمة −2.0 → IC50 = 0.01 μM = 10 nM → pIC50 = 8.0.
- قارن مع المرجعين التجريبيين: نيرماتريلڤير ≈ 31 nM (وسيط ChEMBL) و
  PF-00835231 = 8 nM — الفجوات تُسجَّل كما هي، صادقة، في جدول التدقيق.

In [ ]:
# 6) اختياري — عرض الجدول الثلاثي داخل الدفتر (بعد رفع البيان سابقاً إلى المستودع)
# هذا يحتاج الوصول إلى المجمّد المحلي docs/audit/model-three-tier-p14-results.json
# على GitHub raw بعد الدفع — تعليمياً يكفي الجدول أعلاه.
print("""الخطوات بعد هذا الدفتر:
1) نزّل lab08-manifest.json → ضعه في data/boltz2/lab08-manifest.json بالمستودع
2) شغّل محلياً: python3 scripts/evaluate_three_tier_p14.py
3) الجدول الثلاثي الكامل يُعاد بناؤه وتُجمَّد نسخته في docs/audit/
4) الدرس الكامل: docs/audit/STRUCTURE-TIERS-P14.md""")